# Notebook 2: Learning the Score at One Noise Level

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FloppingCode/modified-langevin-score-matching/blob/main/notebooks/02_single_sigma.ipynb)

**Goal:** Train a small MLP (`SimpleScoreNetwork`) to estimate the score $\nabla_x \log p_\sigma(x)$ at a single fixed noise level $\sigma$ using Denoising Score Matching (DSM). Then sample via vanilla Langevin dynamics and compare with the analytical score.

**Key idea:** DSM trains the network by adding noise $\tilde{x} = x + \sigma \epsilon$ and minimizing
$$\mathcal{L}(\theta) = \mathbb{E}_{x, \epsilon} \left\| s_\theta(\tilde{x}) - \left(-\frac{\epsilon}{\sigma}\right) \right\|^2$$

At a single $\sigma$, this learns the score of the $\sigma$-smoothed distribution. The choice of $\sigma$ critically affects sample quality.

## Setup

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    if os.path.exists("modified-langevin-score-matching"):
        !cd modified-langevin-score-matching && git pull
    else:
        !git clone https://github.com/FloppingCode/modified-langevin-score-matching.git
    sys.path.insert(0, "modified-langevin-score-matching")
else:
    sys.path.insert(0, "..")

In [ ]:
import torch
import matplotlib.pyplot as plt

from dsm import (
    make_dataset, make_dataloader,
    SimpleScoreNetwork,
    single_sigma_dsm_loss,
    vanilla_langevin_dynamics,
    train,
    make_8gaussians_analytical_score,
    save_checkpoint,
)
from dsm.visualization import plot_samples, plot_training_curves, animate_sampling, display_animation

## Configuration

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Dataset
DATASET = "8gaussians"
N_SAMPLES = 10_000
BATCH_SIZE = 512

# Training
SIGMA = 0.3
N_EPOCHS = 200
HIDDEN_DIM = 64
LR = 1e-3

# Sampling
N_GENERATED = 2000
N_STEPS = 2000
STEP_SIZE = 1e-3
SAVE_EVERY = 10  # ~200 frames

## Create Dataset

In [ ]:
dataset = make_dataset(DATASET, n_samples=N_SAMPLES)
data = dataset.tensors[0]
dataloader = make_dataloader(dataset, batch_size=BATCH_SIZE)
print(f"Dataset shape: {data.shape}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(data[:, 0].numpy(), data[:, 1].numpy(), s=1, alpha=0.5)
ax.set_title(f"{DATASET} dataset ({N_SAMPLES} points)")
ax.set_aspect("equal")
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
plt.show()

## Create SimpleScoreNetwork

In [ ]:
model = SimpleScoreNetwork(data_dim=2, hidden_dim=HIDDEN_DIM)
n_params = sum(p.numel() for p in model.parameters())
print(f"SimpleScoreNetwork: {n_params:,} parameters")
print(model)

## Train with Single-Sigma DSM Loss

In [ ]:
history = train(
    model,
    dataloader,
    loss_fn=lambda m, x: single_sigma_dsm_loss(m, x, SIGMA),
    n_epochs=N_EPOCHS,
    lr=LR,
    device=DEVICE,
    log_every=40,
)

In [ ]:
plot_training_curves(history)
plt.show()

## Save Checkpoint

In [ ]:
CHECKPOINT_DIR = "checkpoints" if "google.colab" not in sys.modules else "modified-langevin-score-matching/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

save_checkpoint(
    model,
    os.path.join(CHECKPOINT_DIR, "single_sigma_trained.pt"),
    config={"sigma": SIGMA, "hidden_dim": HIDDEN_DIM, "n_epochs": N_EPOCHS},
    history=history,
)

## Vanilla Langevin Sampling with Learned Score

In [ ]:
learned_samples, learned_traj = vanilla_langevin_dynamics(
    model,
    n_samples=N_GENERATED,
    data_dim=2,
    n_steps=N_STEPS,
    step_size=STEP_SIZE,
    device=DEVICE,
    return_trajectories=True,
    save_every=SAVE_EVERY,
)
print(f"Learned samples: {learned_samples.shape}, trajectory frames: {learned_traj.shape[0]}")

plot_samples(data, learned_samples.cpu(), title=f"Learned Score Sampling (sigma={SIGMA})")
plt.show()

## Vanilla Langevin Sampling with Analytical Score

In [ ]:
analytical_model = make_8gaussians_analytical_score().to(DEVICE)

analytical_samples, analytical_traj = vanilla_langevin_dynamics(
    analytical_model,
    n_samples=N_GENERATED,
    data_dim=2,
    n_steps=N_STEPS,
    step_size=STEP_SIZE,
    device=DEVICE,
    return_trajectories=True,
    save_every=SAVE_EVERY,
)
print(f"Analytical samples: {analytical_samples.shape}")

plot_samples(data, analytical_samples.cpu(), title="Analytical Score Sampling")
plt.show()

## Side-by-Side Comparison: Real | Learned | Analytical

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

data_np = data.numpy()
learned_np = learned_samples.cpu().numpy()
analytical_np = analytical_samples.cpu().numpy()

for ax in axes:
    ax.set_aspect("equal")
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)

axes[0].scatter(data_np[:, 0], data_np[:, 1], s=1, alpha=0.5)
axes[0].set_title("Real Data")

axes[1].scatter(learned_np[:, 0], learned_np[:, 1], s=1, alpha=0.5, color="C1")
axes[1].set_title(f"Learned (sigma={SIGMA})")

axes[2].scatter(analytical_np[:, 0], analytical_np[:, 1], s=1, alpha=0.5, color="C2")
axes[2].set_title("Analytical")

fig.suptitle("Single-Sigma Learned Score vs Analytical Score", fontsize=14)
fig.tight_layout()
plt.show()

## Animation: Learned Model Sampling

In [ ]:
anim_learned = animate_sampling(
    learned_traj,
    real_data=data,
    n_particles=200,
    interval=50,
    title=f"Learned Score Sampling (sigma={SIGMA})",
    trail_length=10,
)
display_animation(anim_learned)

## Sigma Sweep: Effect of Noise Level on Sample Quality

We train separate models at different $\sigma$ values to see how the noise level affects generation quality.

In [ ]:
sigmas_to_try = [0.05, 0.1, 0.3, 0.5]

fig, axes = plt.subplots(1, len(sigmas_to_try), figsize=(5 * len(sigmas_to_try), 5))

for ax, sigma_val in zip(axes, sigmas_to_try):
    # Train a fresh model at this sigma
    model_sweep = SimpleScoreNetwork(data_dim=2, hidden_dim=HIDDEN_DIM).to(DEVICE)
    _ = train(
        model_sweep,
        dataloader,
        loss_fn=lambda m, x, s=sigma_val: single_sigma_dsm_loss(m, x, s),
        n_epochs=N_EPOCHS,
        lr=LR,
        device=DEVICE,
        log_every=N_EPOCHS + 1,  # suppress logging
    )
    
    # Sample
    samples_sweep = vanilla_langevin_dynamics(
        model_sweep,
        n_samples=N_GENERATED,
        data_dim=2,
        n_steps=N_STEPS,
        step_size=STEP_SIZE,
        device=DEVICE,
        return_trajectories=False,
    )
    
    samples_np = samples_sweep.cpu().numpy()
    ax.scatter(samples_np[:, 0], samples_np[:, 1], s=1, alpha=0.5, color="C1")
    ax.set_title(f"sigma = {sigma_val}")
    ax.set_aspect("equal")
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)

fig.suptitle("Effect of Noise Level on Single-Sigma Score Matching", fontsize=14)
fig.tight_layout()
plt.show()

**Observations:**
- **$\sigma = 0.05$:** Score is accurate near the data but nearly zero far away. Particles starting from noise may not find the modes. Works well only if initialized close to the data.
- **$\sigma = 0.1$:** A reasonable middle ground for this dataset, but coverage of the full space is still limited.
- **$\sigma = 0.3$:** Broader score field guides particles from farther away, but the modes appear blurred because the network learns the score of a smoothed distribution.
- **$\sigma = 0.5$:** Heavy smoothing -- modes merge and the generated distribution loses fine structure.

**The dilemma:** No single $\sigma$ is right for both coarse (global) and fine (local) structure. This motivates the multi-scale approach of NCSN (Notebook 03).

## Summary

- A small MLP (~8k params) can learn the score at a fixed noise level via Denoising Score Matching.
- Sample quality depends critically on $\sigma$: too small gives poor coverage, too large gives blurry modes.
- The analytical score provides a useful gold standard for comparison.
- **Key limitation:** A single noise level cannot simultaneously provide global navigation (large $\sigma$) and local refinement (small $\sigma$).

**Next:** In Notebook 03, we train a noise-conditional score network (NCSN) across multiple noise levels and use annealed Langevin dynamics.